In [9]:
%gui tk

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import joblib
import numpy as np

# load saved models and encoder from lab 8
model_dummy = joblib.load("dummy_model.pkl")
model_ohe = joblib.load("onehot_model.pkl")
encoder = joblib.load("encoder.pkl")

feature_columns = model_dummy.feature_names_in_.tolist()

# extract category names from encoder
car_models = encoder.named_transformers_['encoder'].categories_[0]

# GUI setup
root = tk.Tk()
root.title("Car Price Predictor")
root.geometry("400x300")

# --- inputs ---
ttk.Label(root, text="Car Model:").grid(row=0, column=0, padx=5, pady=5, sticky="w")
car_model = tk.StringVar()
ttk.Combobox(root, textvariable=car_model, values=car_models.tolist(), state="readonly").grid(row=0, column=1, pady=5)

ttk.Label(root, text="Mileage (km):").grid(row=1, column=0, padx=5, pady=5, sticky="w")
mileage_var = tk.StringVar()
ttk.Entry(root, textvariable=mileage_var).grid(row=1, column=1, pady=5)

ttk.Label(root, text="Age (years):").grid(row=2, column=0, padx=5, pady=5, sticky="w")
age_var = tk.StringVar()
ttk.Entry(root, textvariable=age_var).grid(row=2, column=1, pady=5)

encoding_method = tk.IntVar(value=0)
ttk.Label(root, text="Encoding Method:").grid(row=3, column=0, padx=5, pady=5, sticky="w")
ttk.Radiobutton(root, text="Dummy Variables", variable=encoding_method, value=0).grid(row=3, column=1, sticky="w")
ttk.Radiobutton(root, text="One-Hot Encoding", variable=encoding_method, value=1).grid(row=4, column=1, sticky="w")

# --- output ---
result_var = tk.StringVar(value="Enter details and click Predict")
ttk.Label(root, textvariable=result_var, font=("Arial", 12, "bold")).grid(row=6, column=0, columnspan=2, pady=20)


# --- prediction function ---
def predict_price():
    try:
        mileage = float(mileage_var.get())
        age = int(age_var.get())
        car = car_model.get()
        method = encoding_method.get()

        if method == 0:
            # dummy model
            # make sure columns match training order
            features = []
            for col in feature_columns:
                if col == "Mileage":
                    features.append(mileage)
                elif col == "Age(yrs)":
                    features.append(age)
                elif col == car:
                    features.append(1)
                else:
                    features.append(0)
            price = model_dummy.predict([features])[0]
        else:
            # One-hot encoding model
            categories = car_models[1:]  # skip first to match training
            car_onehot = [1 if c == car else 0 for c in categories]
            features = np.array(car_onehot + [mileage, age])
            price = model_ohe.predict([features])[0]

        result_var.set(f"Predicted Price: ${price:,.2f}")

    except ValueError:
        messagebox.showerror("Input Error", "Please enter valid numbers.")
    except Exception as e:
        messagebox.showerror("Error", str(e))


# --- Buttons ---
ttk.Button(root, text="Predict", command=predict_price).grid(row=5, column=0, pady=10)
ttk.Button(root, text="Reset", command=lambda: [
    car_model.set(""), mileage_var.set(""), age_var.set(""),
    result_var.set("Enter details and click Predict")
]).grid(row=5, column=1, pady=10)


/Users/dani/Desktop/csc180/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
